# 03 — Feature Engineering

Merge store, oil-price, and date-level holiday information. Create calendar features, lagged sales, and rolling averages. Save the processed data as a Parquet file.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/retail-demand-forecasting")
RAW = PROJECT / "data" / "raw"
OUT = PROJECT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

# Load the Kaggle training data and supporting tables
train = pd.read_csv(RAW / "train.csv", parse_dates=["date"])
stores = pd.read_csv(RAW / "stores.csv")
oil = pd.read_csv(RAW / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(
    RAW / "holidays_events.csv", parse_dates=["date"]
)

print("Training data:", train.shape)

# Add store characteristics
df = train.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

# Prepare daily oil prices
all_dates = pd.date_range(
    train["date"].min(),
    train["date"].max(),
    freq="D"
)

oil_series = (
    oil.dropna(subset=["date"])
       .drop_duplicates("date", keep="last")
       .set_index("date")["dcoilwtico"]
       .sort_index()
       .reindex(all_dates)
)

oil_daily = (
    oil_series.interpolate(method="time")
              .ffill()
              .bfill()
              .rename("dcoilwtico")
              .rename_axis("date")
              .reset_index()
)

df = df.merge(
    oil_daily,
    on="date",
    how="left",
    validate="many_to_one"
)

# Aggregate holiday events by date
holidays = holidays.dropna(subset=["date"]).copy()

holidays["is_transferred"] = (
    holidays["transferred"]
    .astype(str)
    .str.lower()
    .isin(["true", "1"])
    .astype("int8")
)

holiday_summary = (
    holidays.groupby("date")
    .agg(
        holiday_event_rows=("description", "count"),
        transferred_event_rows=("is_transferred", "sum")
    )
    .reset_index()
)

# Create date-level holiday-type indicators
holiday_types = pd.crosstab(
    holidays["date"], holidays["type"]
)

holiday_types.columns = [
    "holiday_type_" + str(c).strip().lower().replace(" ", "_")
    for c in holiday_types.columns
]

holiday_features = holiday_summary.merge(
    holiday_types.reset_index(),
    on="date",
    how="left",
    validate="one_to_one"
)

df = df.merge(
    holiday_features,
    on="date",
    how="left",
    validate="many_to_one"
)

holiday_cols = [
    c for c in df.columns
    if c.startswith("holiday_")
    or c.startswith("transferred_event_")
]

df[holiday_cols] = df[holiday_cols].fillna(0)

# Calendar features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear
df["week_of_year"] = (
    df["date"].dt.isocalendar().week.astype("int16")
)
df["is_weekend"] = (
    df["day_of_week"].isin([5, 6]).astype("int8")
)
df["is_month_start"] = df["date"].dt.is_month_start.astype("int8")
df["is_month_end"] = df["date"].dt.is_month_end.astype("int8")

# Sort before generating historical sales features
df = df.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

group_cols = ["store_nbr", "family"]

# Lagged sales: only earlier observations
for lag in [1, 7, 14, 28]:
    df[f"sales_lag_{lag}"] = (
        df.groupby(group_cols, sort=False)["sales"].shift(lag)
    )

# Rolling averages shifted by one row to avoid target leakage
for window in [7, 28]:
    df[f"sales_rolling_mean_{window}"] = (
        df.groupby(group_cols, sort=False)["sales"]
          .transform(
              lambda s: s.shift(1).rolling(
                  window=window,
                  min_periods=window
              ).mean()
          )
    )

# Save the processed dataset
output = OUT / "train_features.parquet"
df.to_parquet(output, index=False)

print("\nFeature engineering complete!")
print("Output:", output)
print("Shape:", df.shape)
print("Number of features:", len(df.columns))
print("\nMissing values in lag features:")

lag_cols = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_28"
]

print(df[lag_cols].isna().sum())
